# HealthConnect Clinic Week 4 Initial Analysis
## Data Analytics Track, AnalystLab Africa Experience Lab

**Objective:** Understand the appointment dataset structure and quality, and identify
business questions and KPIs relevant to investigating appointment no-shows.

This notebook covers:
1. Dataset overview
2. Data quality assessment
3. Business question exploration
4. KPI identification

In [25]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('HealthConnect_Appointment_Data.csv')
df.head()

,appointment_id,patient_id,gender,age,age_group,appointment_type,booking_date,appointment_date,appointment_day,appointment_time,booking_lead_days,previous_appointments,previous_no_shows,reminder_sent,reminder_channel,distance_to_clinic_km,waiting_time_minutes,appointment_outcome
0,HC-00001,P-1613,Female,39,35-44,Follow-up,2/6/2025,2/18/2025,Tuesday,Afternoon,12,2,0,Yes,WhatsApp,19.3,29.0,No-Show
1,HC-00002,P-0813,Male,31,25-34,Specialist Consultation,2/25/2026,2/27/2026,Friday,Morning,2,6,0,Yes,SMS,14.3,42.0,Attended
2,HC-00003,P-1366,Female,50,45-54,General Consultation,11/16/2025,12/24/2025,Wednesday,Morning,38,5,1,Yes,SMS,11.4,11.0,No-Show
3,HC-00004,P-1031,Male,59,55-64,Follow-up,7/18/2025,8/28/2025,Thursday,Evening,41,3,1,Yes,SMS,7.4,35.0,Attended
4,HC-00005,P-1458,Female,34,25-34,Follow-up,7/9/2025,8/25/2025,Monday,Afternoon,47,3,1,Yes,Email,5.6,27.0,No-Show


## 1. Dataset Overview

Checking the shape of the data, column types, and a quick look at the target variable
(`appointment_outcome`) before going any further.

In [26]:
print("Shape:", df.shape)
print("\nColumn types:\n", df.dtypes)
print("\nUnique patients:", df['patient_id'].nunique())
print("\nOutcome distribution:\n", df['appointment_outcome'].value_counts())
print("\nOutcome distribution (%):\n", df['appointment_outcome'].value_counts(normalize=True).round(3) * 100)

Shape: (5000, 18)

Column types:
 appointment_id            object
patient_id                object
gender                    object
age                        int64
age_group                 object
appointment_type          object
booking_date              object
appointment_date          object
appointment_day           object
appointment_time          object
booking_lead_days          int64
previous_appointments      int64
previous_no_shows          int64
reminder_sent             object
reminder_channel          object
distance_to_clinic_km    float64
waiting_time_minutes     float64
appointment_outcome       object
dtype: object

Unique patients: 1696

Outcome distribution:
 appointment_outcome
No-Show      2423
Attended     2314
Cancelled     263
Name: count, dtype: int64

Outcome distribution (%):
 appointment_outcome
No-Show      48.5
Attended     46.3
Cancelled     5.3
Name: proportion, dtype: float64


## 2. Data Quality Assessment

Checking for missing values, duplicate keys, and a couple of logical consistency checks
(e.g. can `previous_no_shows` exceed `previous_appointments`?).

In [27]:
# Missing values
print("Missing values per column:\n", df.isnull().sum())

# Duplicate primary key
print("\nDuplicate appointment_id rows:", df['appointment_id'].duplicated().sum())

# Logical check: no-shows shouldn't exceed total previous appointments
invalid_history = (df['previous_no_shows'] > df['previous_appointments']).sum()
print("Rows where previous_no_shows > previous_appointments:", invalid_history)

Missing values per column:
 appointment_id              0
patient_id                  0
gender                      0
age                         0
age_group                   0
appointment_type            0
booking_date                0
appointment_date            0
appointment_day             0
appointment_time            0
booking_lead_days           0
previous_appointments       0
previous_no_shows           0
reminder_sent               0
reminder_channel         1366
distance_to_clinic_km      90
waiting_time_minutes       60
appointment_outcome         0
dtype: int64

Duplicate appointment_id rows: 0
Rows where previous_no_shows > previous_appointments: 0


In [28]:
# Confirm reminder_channel blanks line up with reminder_sent = 'No'
# (i.e. these aren't true missing values, they're "not applicable")
check = df[df['reminder_channel'].isnull()]['reminder_sent'].value_counts()
print("reminder_sent values where reminder_channel is blank:\n", check)

reminder_sent values where reminder_channel is blank:
 reminder_sent
No    1366
Name: count, dtype: int64


Confirms the `reminder_channel` blanks aren't a data quality issue they're "not
applicable" rows because no reminder was sent. Recoding them so they don't get
counted as missing data later.

In [29]:
df_clean = df.copy()
df_clean['reminder_channel'] = df_clean['reminder_channel'].fillna('Not Applicable')

# distance_to_clinic_km and waiting_time_minutes have genuine small gaps — flag and impute with median
for col in ['distance_to_clinic_km', 'waiting_time_minutes']:
    df_clean[f'{col}_was_missing'] = df_clean[col].isnull()
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print(df_clean[['reminder_channel', 'distance_to_clinic_km_was_missing', 'waiting_time_minutes_was_missing']].isnull().sum())
print("\nRemaining missing values:\n", df_clean.isnull().sum().sum())

reminder_channel                     0
distance_to_clinic_km_was_missing    0
waiting_time_minutes_was_missing     0
dtype: int64

Remaining missing values:
 0


Saving the cleaned version separately, per the resource instructions never overwrite
the original file.

In [30]:
df_clean.to_csv('HealthConnect_Appointment_Data_cleaned.csv', index=False)
print("Saved cleaned dataset:", df_clean.shape)

Saved cleaned dataset: (5000, 20)


## 3. Business Question Exploration

Quick group-by checks against `appointment_outcome` to see which variables actually
show a relationship with no-shows before committing to business questions.

In [31]:
def no_show_rate(group):
    return (group['appointment_outcome'] == 'No-Show').mean().round(3)

print("No-show rate by reminder_sent:\n", df_clean.groupby('reminder_sent').apply(no_show_rate))
print("\nNo-show rate by reminder_channel:\n", df_clean.groupby('reminder_channel').apply(no_show_rate))
print("\nNo-show rate by appointment_type:\n", df_clean.groupby('appointment_type').apply(no_show_rate))
print("\nNo-show rate by appointment_time:\n", df_clean.groupby('appointment_time').apply(no_show_rate))
print("\nNo-show rate by age_group:\n", df_clean.groupby('age_group').apply(no_show_rate))

No-show rate by reminder_sent:
 reminder_sent
No     0.514
Yes    0.474
dtype: float64



No-show rate by reminder_channel:
 reminder_channel
Email             0.484
Not Applicable    0.514
SMS               0.458
WhatsApp          0.498
dtype: float64

No-show rate by appointment_type:
 appointment_type
Diagnostic Test            0.497
Follow-up                  0.512
General Consultation       0.466
Specialist Consultation    0.474
dtype: float64

No-show rate by appointment_time:
 appointment_time
Afternoon    0.484
Evening      0.498
Morning      0.481
dtype: float64

No-show rate by age_group:
 age_group
18-24    0.502
25-34    0.507
35-44    0.484
45-54    0.480
55-64    0.507
65+      0.451
dtype: float64


C:\Users\brand\AppData\Local\Temp\ipykernel_1816\3664572750.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print("No-show rate by reminder_sent:\n", df_clean.groupby('reminder_sent').apply(no_show_rate))
C:\Users\brand\AppData\Local\Temp\ipykernel_1816\3664572750.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print("\nNo-show rate by reminder_channel:\n", df_clean.groupby('reminder_channel').apply(no_show_r

In [32]:
# Booking lead time
df_clean['lead_bucket'] = pd.cut(df_clean['booking_lead_days'], [-1, 3, 7, 14, 30, 60],
                                   labels=['0-3', '4-7', '8-14', '15-30', '31-60'])
print("No-show rate by booking lead time (days):\n", df_clean.groupby('lead_bucket', observed=True).apply(no_show_rate))

# Previous no-shows
df_clean['prev_ns_bucket'] = pd.cut(df_clean['previous_no_shows'], [-1, 0, 1, 2, 20],
                                      labels=['0', '1', '2', '3+'])
print("\nNo-show rate by previous no-shows:\n", df_clean.groupby('prev_ns_bucket', observed=True).apply(no_show_rate))

# Distance
df_clean['dist_bucket'] = pd.cut(df_clean['distance_to_clinic_km'], [0, 5, 10, 20, 50],
                                   labels=['0-5km', '5-10km', '10-20km', '20-50km'])
print("\nNo-show rate by distance to clinic:\n", df_clean.groupby('dist_bucket', observed=True).apply(no_show_rate))

No-show rate by booking lead time (days):
 lead_bucket
0-3      0.248
4-7      0.307
8-14     0.336
15-30    0.432
31-60    0.605
dtype: float64

No-show rate by previous no-shows:
 prev_ns_bucket
0     0.435
1     0.535
2     0.594
3+    0.688
dtype: float64

No-show rate by distance to clinic:
 dist_bucket
0-5km      0.465
5-10km     0.468
10-20km    0.494
20-50km    0.578
dtype: float64


C:\Users\brand\AppData\Local\Temp\ipykernel_1816\3593125733.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print("No-show rate by booking lead time (days):\n", df_clean.groupby('lead_bucket', observed=True).apply(no_show_rate))
C:\Users\brand\AppData\Local\Temp\ipykernel_1816\3593125733.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print("\nNo-show rate by previous no-shows:\n", df_clean.groupby('prev_ns_b

**Reading the output:** `lead_bucket` and `prev_ns_bucket` should show the clearest
upward trend in no-show rate these are the strongest candidates for business
questions. `reminder_sent`, `appointment_type`, `appointment_time`, and `age_group`
should come back fairly flat, which is itself a useful (if less exciting) finding.

## 4. KPI Summary

Five potential KPIs, each tied to a business question. At this stage these are just
identified and justified — not yet calculated as final dashboard metrics.

| KPI | Business Question |
|---|---|
| No-show rate by booking lead time band | Does lead time affect no-show risk? |
| No-show rate by prior no-show count band | Does history predict future no-shows? |
| No-show rate by distance band | Does distance affect attendance? |
| Reminder effectiveness rate (by channel) | Do reminders reduce no-shows? |
| No-show rate by appointment type / time / age group | Do these factors affect attendance? |

**Reading:** reminder_sent, reminder_channel, appointment_type, appointment_time, and
age_group all show weak, flat relationships with no-show rate. Every category sits
within about 5 percentage points of the others (e.g. reminder sent 47.4% vs not sent
51.4%; age groups range 45.1% to 50.7%). None of these variables look like strong
standalone predictors on their own.

This is a useful finding, not a dead end. It suggests reminders may not be acting
independently of other factors (e.g. they might be sent regardless of a patient's
actual risk level), which is worth testing further rather than concluding reminders
"don't work."

**Reading:** These three variables show a much clearer relationship with no-show rate.

- **Booking lead time** is the strongest driver: no-show rate more than doubles from
  24.8% (booked 0-3 days ahead) to 60.5% (booked 31-60 days ahead), a steady,
  near-linear increase across every band.
- **Prior no-shows** shows the same pattern: 43.5% for patients with no history of
  missing appointments, rising to 68.8% for patients with 3 or more prior no-shows.
- **Distance to clinic** shows a moderate effect: fairly flat from 0-20km (46-49%),
  then a jump to 57.8% beyond 20km.

These three variables, lead time, prior no-show history, and distance, are the
clearest candidates to prioritise in later-stage analysis and any future predictive
model, since they show a consistent, explainable relationship with the outcome rather
than noise.